# Task 1 — Sentiment Analysis

Compare TF-IDF models with a pretrained transformer on ONE shared test split. Report accuracy, per-class precision/recall/F1 and the confusion matrix, then save the chosen model for the app.

*Roman Urdu: Saare models ek hi test set pe compare honge, phir best wala app mein jayega.*

In [ ]:
# All imports in one place
import sys
sys.path.append("..")        # project root -> src/
sys.path.append("../app")    # app/ -> text_cleaning.py

import importlib
import json
import re
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import nltk
import pandas as pd
import sklearn
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight

from src import config, preprocessing
import text_cleaning

importlib.reload(preprocessing)
importlib.reload(text_cleaning)      # pick up edits to app/text_cleaning.py

pre_processed = preprocessing.load_pre_processed()
pre_processed.info()

In [ ]:
# Settings used everywhere below
RANDOM_STATE = 42
TEST_SIZE = 0.3
LABELS = ["negative", "neutral", "positive"]
RATING_TO_LABEL = {1.0: "negative", 2.0: "negative", 3.0: "neutral", 4.0: "positive", 5.0: "positive"}
APP_DIR = Path("../app")

y = pre_processed["reviews.rating"].map(RATING_TO_LABEL)
X = pre_processed["reviews.text"].astype(str)                                     # text only (TF-IDF v1)
X_full = pre_processed["reviews.title"].fillna("").astype(str) + ". " + X         # title + text (v2, transformer)

# ONE split, reused by every model, so all models are tested on the same reviews
train_idx, test_idx = train_test_split(
    pre_processed.index, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train, X_test = X.loc[train_idx], X.loc[test_idx]
X_full_train, X_full_test = X_full.loc[train_idx], X_full.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print(y_train.value_counts(normalize=True).round(3))

# One helper to score every model the same way; results feed the comparison table
results = {}

def evaluate(name, y_true, y_pred):
    f1 = f1_score(y_true, y_pred, labels=LABELS, average=None)
    results[name] = {
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "macro_f1": round(f1.mean(), 3),
        "f1_negative": round(f1[0], 3),
        "f1_neutral": round(f1[1], 3),
        "f1_positive": round(f1[2], 3),
    }
    print(f"=== {name}")
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))
    print(confusion_matrix(y_true, y_pred, labels=LABELS))

In [ ]:
# Step 1: remove HTML (same function the app uses, imported from app/text_cleaning.py)
from text_cleaning import remove_html_tags

print(remove_html_tags("<b>Great</b> tablet<br>for kids"))

In [ ]:
# Step 2: lowercase, letters only, drop single letters and extra spaces
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\b[a-z]\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()

print(clean_text("Great!! 10/10, I'd buy it again"))

In [ ]:
# Step 3: remove stopwords (the list is built ONCE, not on every call)
STOP_WORDS = set(stopwords.words("english"))

def remove_stopwords(text):
    return " ".join(w for w in text.split() if w not in STOP_WORDS)

print(remove_stopwords("this is not a good tablet"))   # note: "not" is removed here (fixed in v2)

In [ ]:
# Step 4: lemmatize, then run all 4 steps ONCE on train and test
lemmatizer = WordNetLemmatizer()

def lemmatize(text):
    return " ".join(lemmatizer.lemmatize(w) for w in text.split())

def clean_v1(text):
    return lemmatize(remove_stopwords(clean_text(remove_html_tags(text))))

X_train_clean = X_train.apply(clean_v1)
X_test_clean = X_test.apply(clean_v1)
X_train_clean.head()

In [ ]:
# TF-IDF: turn text into word scores, then show the most typical words per class
def top_terms(label, tfidf_matrix, vectorizer, top_n=5):
    """Words with the highest average TF-IDF score among training reviews of one label."""
    rows = (y_train == label).to_numpy()
    mean_scores = tfidf_matrix[rows].mean(axis=0).A1
    return pd.Series(mean_scores, index=vectorizer.get_feature_names_out()).nlargest(top_n)

def fit_tfidf(vectorizer):
    """Fit on train only, transform test, print shapes and top words per class."""
    X_tr = vectorizer.fit_transform(X_train_clean)
    X_te = vectorizer.transform(X_test_clean)
    print("Train:", X_tr.shape, "Test:", X_te.shape)
    for label in LABELS:
        print(f"\nTop {label} words:\n", top_terms(label, X_tr, vectorizer))
    return X_tr, X_te

# First try: single words, 5,000 most frequent
X_train_tfidf, X_test_tfidf = fit_tfidf(TfidfVectorizer(stop_words="english", max_features=5000))

In [ ]:
# Better settings: word pairs, ignore words seen once, softer counts (these are used by the models below)
X_train_tfidf, X_test_tfidf = fit_tfidf(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)
)

In [ ]:
# Model 1: Naive Bayes (sample weights so rare classes count more)
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train, sample_weight=compute_sample_weight("balanced", y_train))
evaluate("TF-IDF v1 + Naive Bayes", y_test, nb_model.predict(X_test_tfidf))

In [ ]:
# Model 2: Logistic Regression
logreg_v1 = LogisticRegression(class_weight="balanced", max_iter=1000)
logreg_v1.fit(X_train_tfidf, y_train)
evaluate("TF-IDF v1 + LogReg", y_test, logreg_v1.predict(X_test_tfidf))

In [ ]:
# Model 3: Linear SVM
svm_model = LinearSVC(class_weight="balanced")
svm_model.fit(X_train_tfidf, y_train)
evaluate("TF-IDF v1 + LinearSVC", y_test, svm_model.predict(X_test_tfidf))

## Improving the model

Testing the first saved model in the app showed it handles positive reviews well but fails on negative and neutral ones
(e.g. *"Worst customer service I have ever experienced"* → positive). Reasons:

1. **Very few negative / neutral examples**: 93% of reviews are positive, so words like "price", "quality" and "customer service" mostly appear in positive reviews.
2. **Cleaning removed key words**: the NLTK stopword list drops "not", "no", "too", "very", so *"price is too high"* became *"price high"*.
3. **TF-IDF only counts words**, it does not understand meaning.

What we try, all on the **same test split** (`test_size=0.3`, `random_state=42`, stratified):

| Model | Change |
|---|---|
| TF-IDF v1 (cells above) | stopwords + lemmatizing, text only → macro-F1 ≈ 0.52 (Logistic Regression) |
| **TF-IDF v2** | keep negation words, expand "n't" → "not", no lemmatizing, **title + text**, `C=10` |
| **Pretrained transformer (zero-shot)** | `cardiffnlp/twitter-roberta-base-sentiment-latest`, already trained for negative/neutral/positive, no training on our data |

We compare by **macro-F1** (accuracy hides the small classes) and by a few hand-written reviews.

*Roman Urdu: Har model ko same test set pe macro-F1 se compare karo, phir jeetne wala app mein jayega.*

In [ ]:
# Hand-written reviews, including the ones the first app got wrong
HAND_TESTS = [
    "price is too high and quality is really bad",
    "This product broke after two days of light use.",
    "Worst customer service I have ever experienced.",
    "Terrible interface, nothing works as advertised",
    "not going to recommend to anyone",
    "It arrived damaged and missing key parts",
    "It is a standard blue shirt made of cotton",
    "It works, but the screen is dimmer than I expected for the price.",
    "Exceeded my expectations, absolutely love it!",
]

### TF-IDF v2

In [ ]:
tfidf_v2 = Pipeline([
    ("clean", FunctionTransformer(text_cleaning.preprocess)),   # keeps not / no / too
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)),
    ("model", LogisticRegression(class_weight="balanced", C=10, max_iter=2000)),
])
tfidf_v2.fit(X_full_train, y_train)          # raw title + text in; the Pipeline cleans it itself
evaluate("TF-IDF v2 + LogReg", y_test, tfidf_v2.predict(X_full_test))

### Pretrained transformer (zero-shot)
No training: the model was already trained on ~124M tweets for negative / neutral / positive.
Needs `transformers` and `torch`: if the import fails, run `%pip install transformers torch` and restart the kernel.
The first run downloads the model (~500 MB). Predicting the 8,359 test reviews takes a few minutes on a Mac;
set `EVAL_SAMPLE = 2000` for a quicker (approximate) check.

In [ ]:
# Kept here (not at the top) so the notebook still runs if torch is not installed
import torch
from transformers import pipeline as hf_pipeline

TRANSFORMER_ID = "cardiffnlp/twitter-roberta-base-sentiment-latest"
EVAL_SAMPLE = None          # e.g. 2000 for a faster check; None = full test set

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
transformer = hf_pipeline("text-classification", model=TRANSFORMER_ID, device=device)

X_eval, y_eval = X_full_test, y_test
if EVAL_SAMPLE:
    X_eval = X_full_test.sample(EVAL_SAMPLE, random_state=RANDOM_STATE)
    y_eval = y_test.loc[X_eval.index]

out = transformer(X_eval.tolist(), batch_size=32, truncation=True, max_length=512)
evaluate("Transformer zero-shot (roberta)", y_eval, [d["label"].lower() for d in out])

if EVAL_SAMPLE:   # score TF-IDF v2 on the same subset, so the comparison is fair
    evaluate("TF-IDF v2 + LogReg (same sample)", y_eval, tfidf_v2.predict(X_eval))

### Compare

In [ ]:
comparison = pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)
display(comparison)

hand = pd.DataFrame({
    "review": HAND_TESTS,
    "TF-IDF v2": tfidf_v2.predict(HAND_TESTS),
    "Transformer": [d["label"].lower() for d in transformer(HAND_TESTS, truncation=True, max_length=512)],
})
display(hand)

### Choose and save the model for the app
By default the model with the higher macro-F1 wins. Look at the hand-written tests too:
if the scores are close, prefer the model that gets those right, and set `CHOICE` yourself.
Write one sentence on why you chose it.

The cell writes `app/model_config.json` (which model the app loads) and `app/requirements.txt` (with your exact versions).

In [ ]:
CHOICE = "transformer"    # None = pick by macro-F1, or set "tfidf" / "transformer"

tfidf_name = "TF-IDF v2 + LogReg (same sample)" if EVAL_SAMPLE else "TF-IDF v2 + LogReg"
if CHOICE is None:
    CHOICE = "transformer" if results["Transformer zero-shot (roberta)"]["macro_f1"] > results[tfidf_name]["macro_f1"] else "tfidf"
print("Serving:", CHOICE)

# The TF-IDF model is always saved (small, useful as a fallback)
joblib.dump(tfidf_v2, APP_DIR / "sentiment_model.joblib")
joblib.dump(tfidf_v2, config.MODELS_DIR / "sentiment_tfidf_v2_logreg.joblib")

if CHOICE == "transformer":
    import transformers
    model_config = {"type": "transformer", "model_id": TRANSFORMER_ID}
    requirements = f"gradio\ntransformers=={transformers.__version__}\ntorch\n"
else:
    model_config = {"type": "tfidf"}
    requirements = (
        "gradio\n"
        f"scikit-learn=={sklearn.__version__}\n"
        f"nltk=={nltk.__version__}\n"
        f"joblib=={joblib.__version__}\n"
    )

(APP_DIR / "model_config.json").write_text(json.dumps(model_config))
(APP_DIR / "requirements.txt").write_text(requirements)
comparison.to_csv(config.METRICS_DIR / "sentiment_model_comparison.csv")

print("model_config.json:", model_config)
print("requirements.txt:\n" + requirements)